# Trout Age4 Model Comparison

This notebook compares age prediction models for readable trout scale images using the collapsed age labels:

- `0+`
- `1+`
- `2+`
- `3+` = original ages `3`, `4`, and `5`

Main comparisons:
- length/weight only;
- texture feature only;
- texture + length/weight;
- optional CNN image model;
- optional CNN + texture + length/weight fusion model.

All reported splits use `fish_key` grouping so images from the same fish do not appear in both train and test.

## 1. Setup

Run `trout_new_dataset_eda.ipynb` and `trout_texture_feature_extraction.ipynb` first. For final modeling, the texture notebook should be run with `RUN_SAMPLE=False`.

In [ ]:
from __future__ import annotations

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
FEATURE_OUTPUT_DIR = CODE_DIR / "feature_outputs"
MODEL_OUTPUT_DIR = CODE_DIR / "model_outputs"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_MASTER_TEXTURE_CSV = FEATURE_OUTPUT_DIR / "master_with_texture_features_full.csv"
SAMPLE_MASTER_TEXTURE_CSV = FEATURE_OUTPUT_DIR / "master_with_texture_features_sample.csv"

SEED = 100
TEST_SIZE = 0.2
CLASS_NAMES = ["0+", "1+", "2+", "3+"]

random.seed(SEED)
np.random.seed(SEED)

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("FEATURE_OUTPUT_DIR:", FEATURE_OUTPUT_DIR)

## 2. Load Texture Master Table

In [ ]:
if FULL_MASTER_TEXTURE_CSV.exists():
    MASTER_TEXTURE_CSV = FULL_MASTER_TEXTURE_CSV
elif SAMPLE_MASTER_TEXTURE_CSV.exists():
    MASTER_TEXTURE_CSV = SAMPLE_MASTER_TEXTURE_CSV
    print("WARNING: using sample texture table. Run full feature extraction for final results.")
else:
    raise FileNotFoundError(
        "Missing texture master table. Run trout_texture_feature_extraction.ipynb first."
    )

master_texture_df = pd.read_csv(MASTER_TEXTURE_CSV)
print("Loaded:", MASTER_TEXTURE_CSV)
print("master_texture_df:", master_texture_df.shape)

required_cols = {"path", "scale_id", "fish_key", "age4", "has_texture_features", "length_mm", "weight_g"}
missing_cols = required_cols - set(master_texture_df.columns)
if missing_cols:
    raise ValueError(f"Missing columns: {sorted(missing_cols)}")

model_df = master_texture_df[
    master_texture_df["age4"].notna()
    & master_texture_df["has_texture_features"].astype(bool)
].copy()
model_df["age4"] = model_df["age4"].astype(int)
model_df["path_exists"] = model_df["path"].map(lambda p: Path(str(p)).exists())

print("model_df:", model_df.shape)
print("path missing:", int((~model_df["path_exists"]).sum()))
print("age4 counts:")
display(model_df["age4"].value_counts().sort_index())
print("unique fish:", model_df["fish_key"].nunique())
display(model_df.head())

## 3. Feature Sets

The comparison uses the same train/test fish split for every model.

In [ ]:
metadata_cols = {
    "path", "file", "scale_id", "fish_key", "river", "point", "fish_id", "image_idx", "relative_path",
    "sampling_date", "age_est", "obs", "label", "label_source", "known_bad", "age4", "id_old",
    "length_mm", "weight_g", "split", "is_readable_labeled", "is_age4_labeled", "path_exists", "has_texture_features",
}

texture_cols = [
    c for c in model_df.columns
    if c not in metadata_cols and pd.api.types.is_numeric_dtype(model_df[c])
]
length_weight_cols = ["length_mm", "weight_g"]
combined_cols = texture_cols + length_weight_cols

print("texture features:", len(texture_cols))
print("length/weight features:", length_weight_cols)
print("combined features:", len(combined_cols))
print(texture_cols[:20])

In [ ]:
if len(model_df) < 50:
    raise ValueError("Not enough age4-labeled rows with texture features. Use full texture extraction first.")
if model_df["fish_key"].nunique() < 10:
    raise ValueError("Not enough unique fish for fish-level splitting.")

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(model_df, model_df["age4"], groups=model_df["fish_key"]))
train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

fish_overlap = sorted(set(train_df["fish_key"]) & set(test_df["fish_key"]))
print("train:", train_df.shape, train_df["age4"].value_counts().sort_index().to_dict())
print("test :", test_df.shape, test_df["age4"].value_counts().sort_index().to_dict())
print("Fish overlap:", len(fish_overlap))
assert len(fish_overlap) == 0

## 4. RandomForest Baselines

These baselines answer whether handcrafted texture features add information beyond fish length and weight.

In [ ]:
def evaluate_predictions(name: str, y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    print("\n===", name, "===")
    print("Accuracy:", round(acc, 4))
    print("Balanced accuracy:", round(bal_acc, 4))
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
    return {
        "model": name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    }


def fit_random_forest_baseline(name: str, feature_cols: list[str]) -> tuple[object, dict, np.ndarray]:
    clf = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=500,
            random_state=SEED,
            class_weight="balanced_subsample",
            n_jobs=-1,
        ),
    )
    clf.fit(train_df[feature_cols], train_df["age4"])
    pred = clf.predict(test_df[feature_cols])
    metrics = evaluate_predictions(name, test_df["age4"], pred)
    return clf, metrics, pred

results = []
models = {}
predictions = {}

for name, cols in [
    ("length_weight_only", length_weight_cols),
    ("texture_only", texture_cols),
    ("texture_plus_length_weight", combined_cols),
]:
    clf, metrics, pred = fit_random_forest_baseline(name, cols)
    models[name] = clf
    predictions[name] = pred
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
display(results_df)

In [ ]:
results_path = MODEL_OUTPUT_DIR / "age4_baseline_results.csv"
results_df.to_csv(results_path, index=False)
print("saved:", results_path)

for name, pred in predictions.items():
    cm = confusion_matrix(test_df["age4"], pred, labels=[0, 1, 2, 3])
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = MODEL_OUTPUT_DIR / f"confusion_matrix_{name}.csv"
    cm_df.to_csv(cm_path)
    print("saved:", cm_path)
    display(cm_df)

## 5. Feature Importance for Texture + Length/Weight

This is useful for explaining what the handcrafted features are contributing.

In [ ]:
best_rf = models["texture_plus_length_weight"].named_steps["randomforestclassifier"]
importance_df = pd.DataFrame({
    "feature": combined_cols,
    "importance": best_rf.feature_importances_,
}).sort_values("importance", ascending=False)

importance_path = MODEL_OUTPUT_DIR / "texture_plus_length_weight_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)
print("saved:", importance_path)
display(importance_df.head(30))

## 6. Optional CNN Setup

The next cells test whether image pixels improve over handcrafted features. They require `torch` and `torchvision` in the active Jupyter kernel.

Two CNN variants are included:
- image-only ImageNet ResNet18;
- fusion model: ResNet18 image embedding + texture/length/weight tabular branch.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torchvision.transforms as T
    import torchvision.models as models_tv
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from tqdm.auto import tqdm
    TORCH_AVAILABLE = True
except ImportError as exc:
    TORCH_AVAILABLE = False
    print("Skipping CNN cells because torch/torchvision is unavailable:", exc)

if TORCH_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_CUDA = device.type == "cuda"
    NUM_WORKERS = 0
    PIN_MEMORY = USE_CUDA
    BATCH_SIZE = 64
    CNN_EPOCHS = 10
    LR = 1e-4
    print("device:", device)
    print("BATCH_SIZE:", BATCH_SIZE, "CNN_EPOCHS:", CNN_EPOCHS)

In [ ]:
if TORCH_AVAILABLE:
    def set_torch_seed(seed: int = SEED) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    set_torch_seed()

    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    class TroutImageTabularDataset(Dataset):
        def __init__(self, df: pd.DataFrame, tabular_cols: list[str] | None = None, transform=None, imputer=None):
            self.df = df.reset_index(drop=True)
            self.tabular_cols = tabular_cols or []
            self.transform = transform
            self.y = self.df["age4"].astype(int).to_numpy()
            self.imputer = imputer
            if self.tabular_cols:
                X = self.df[self.tabular_cols].to_numpy(dtype=np.float32)
                if self.imputer is not None:
                    X = self.imputer.transform(X).astype(np.float32)
                self.X_tab = X
            else:
                self.X_tab = None

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            image = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            y = int(self.y[idx])
            if self.X_tab is None:
                return image, y
            return image, torch.tensor(self.X_tab[idx], dtype=torch.float32), y

    def make_resnet18_backbone():
        try:
            weights = models_tv.ResNet18_Weights.DEFAULT
            model = models_tv.resnet18(weights=weights)
        except AttributeError:
            model = models_tv.resnet18(pretrained=True)
        model.fc = nn.Identity()
        return model

    class ImageOnlyClassifier(nn.Module):
        def __init__(self, n_classes=4):
            super().__init__()
            self.backbone = make_resnet18_backbone()
            self.head = nn.Linear(512, n_classes)

        def forward(self, x):
            return self.head(self.backbone(x))

    class ImageTabularFusionClassifier(nn.Module):
        def __init__(self, n_tabular: int, n_classes=4):
            super().__init__()
            self.backbone = make_resnet18_backbone()
            self.tabular_net = nn.Sequential(
                nn.Linear(n_tabular, 64),
                nn.ReLU(),
                nn.BatchNorm1d(64),
                nn.Dropout(0.2),
                nn.Linear(64, 32),
                nn.ReLU(),
            )
            self.head = nn.Sequential(
                nn.Linear(512 + 32, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, n_classes),
            )

        def forward(self, image, tabular):
            image_feat = self.backbone(image)
            tab_feat = self.tabular_net(tabular)
            return self.head(torch.cat([image_feat, tab_feat], dim=1))

    print("CNN classes ready")

## 7. Optional CNN Training Helpers

In [ ]:
if TORCH_AVAILABLE:
    def class_weight_tensor(y: pd.Series) -> torch.Tensor:
        counts = y.value_counts().reindex([0, 1, 2, 3], fill_value=0).astype(float)
        weights = counts.sum() / (len(counts) * counts.clip(lower=1))
        return torch.tensor(weights.to_numpy(), dtype=torch.float32, device=device)

    def train_image_only_model(epochs: int = CNN_EPOCHS):
        train_ds = TroutImageTabularDataset(train_df, transform=train_transform)
        test_ds = TroutImageTabularDataset(test_df, transform=eval_transform)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = ImageOnlyClassifier().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age4"]))

        for epoch in range(epochs):
            model.train()
            losses = []
            for images, y in tqdm(train_loader, desc=f"image-only {epoch + 1}/{epochs}"):
                images = images.to(device)
                y = y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images), y)
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")
        return model, test_loader

    def train_fusion_model(tabular_cols: list[str], epochs: int = CNN_EPOCHS):
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        train_tab = scaler.fit_transform(imputer.fit_transform(train_df[tabular_cols].to_numpy(dtype=np.float32)))
        test_tab = scaler.transform(imputer.transform(test_df[tabular_cols].to_numpy(dtype=np.float32)))

        class FixedTabularDataset(Dataset):
            def __init__(self, df, tab_array, transform):
                self.df = df.reset_index(drop=True)
                self.tab_array = tab_array.astype(np.float32)
                self.transform = transform
                self.y = self.df["age4"].astype(int).to_numpy()

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                image = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
                image = self.transform(image)
                return image, torch.tensor(self.tab_array[idx], dtype=torch.float32), int(self.y[idx])

        train_ds = FixedTabularDataset(train_df, train_tab, train_transform)
        test_ds = FixedTabularDataset(test_df, test_tab, eval_transform)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = ImageTabularFusionClassifier(n_tabular=len(tabular_cols)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age4"]))

        for epoch in range(epochs):
            model.train()
            losses = []
            for images, tabular, y in tqdm(train_loader, desc=f"fusion {epoch + 1}/{epochs}"):
                images = images.to(device)
                tabular = tabular.to(device)
                y = y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images, tabular), y)
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")
        return model, test_loader

    @torch.no_grad()
    def evaluate_image_model(model, loader, name: str, fusion: bool = False) -> dict:
        model.eval()
        y_true, y_pred = [], []
        for batch in tqdm(loader, desc=f"evaluate {name}"):
            if fusion:
                images, tabular, y = batch
                logits = model(images.to(device), tabular.to(device))
            else:
                images, y = batch
                logits = model(images.to(device))
            pred = logits.argmax(dim=1).detach().cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(y.numpy().tolist())
        return evaluate_predictions(name, y_true, y_pred)

## 8. Optional: Run CNN Image-Only and Fusion Models

These cells are longer-running. Use them after the RandomForest baselines look reasonable.

In [ ]:
# Uncomment when ready.
# image_model, image_test_loader = train_image_only_model(epochs=CNN_EPOCHS)
# image_metrics = evaluate_image_model(image_model, image_test_loader, "cnn_image_only", fusion=False)
# results.append(image_metrics)
# pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)

In [ ]:
# Uncomment when ready.
# fusion_model, fusion_test_loader = train_fusion_model(combined_cols, epochs=CNN_EPOCHS)
# fusion_metrics = evaluate_image_model(fusion_model, fusion_test_loader, "cnn_texture_length_weight_fusion", fusion=True)
# results.append(fusion_metrics)
# pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)

## 9. Interpretation

If `texture_plus_length_weight` beats both `texture_only` and `length_weight_only`, then the handcrafted image descriptors contain complementary information to fish morphology.

If `cnn_texture_length_weight_fusion` beats `cnn_image_only`, then the handcrafted texture and morphology features add useful information beyond ImageNet-pretrained image features.